In [47]:
import torch
import torch as t
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
import torch.nn.functional as F
import torch.optim as optim

#sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine

In [34]:
X,y = load_wine(return_X_y=True)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=7)
scaler = StandardScaler()
X_train_sca = scaler.fit_transform(X_train)
X_test_sca = scaler.transform(X_test)
X_train_sca_tensor = t.tensor(X_train_sca).float()
X_test_sca_tensor = t.tensor(X_test_sca).float()
y_train_tensor = t.tensor(y_train)
y_test_tensor = t.tensor(y_test)

In [35]:
wine = load_wine()
print(len(wine.feature_names),wine.target_names)

13 ['class_0' 'class_1' 'class_2']


In [36]:
wine = TensorDataset(X_train_sca_tensor,y_train_tensor)
loader = DataLoader(dataset=wine,batch_size=32,shuffle=True)

In [37]:
class BCNet(nn.Module):
    def __init__(self):
        super(BCNet,self).__init__()
        self.fc1 = nn.Linear(13,512)
        self.fc2 = nn.Linear(512,256)
        self.fc3 = nn.Linear(256,128)
        self.fc4 = nn.Linear(128,3)

    def forward(self,x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x


In [38]:
model = BCNet()
creterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [42]:
epochs = 1000
for epoch in range(epochs):
    running_loss = 0
    model.train()
    for batchX,batchy in loader:
        optimizer.zero_grad()
        outputs = model(batchX)#forward propagation
        loss = creterion(outputs,batchy)#loss
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print('epoch ',epoch,' loss ',running_loss/len(loader))

epoch  0  loss  4.289404866852919e-08
epoch  1  loss  3.714645906427449e-08
epoch  2  loss  4.002025368876616e-08
epoch  3  loss  4.5767838940946605e-08
epoch  4  loss  3.9062322443328415e-08
epoch  5  loss  4.0020253777584e-08
epoch  6  loss  3.927519571789162e-08
epoch  7  loss  3.7359333004971515e-08
epoch  8  loss  3.4485537625528197e-08
epoch  9  loss  3.73593323388377e-08
epoch  10  loss  4.693864514848656e-08
epoch  11  loss  3.278254823158022e-08
epoch  12  loss  3.374047885529308e-08
epoch  13  loss  3.4698410900091404e-08
epoch  14  loss  3.3740479210564443e-08
epoch  15  loss  3.5656343655432465e-08
epoch  16  loss  3.778508030904959e-08
epoch  17  loss  3.491128595101145e-08
epoch  18  loss  3.491128448551706e-08
epoch  19  loss  3.586921621945294e-08
epoch  20  loss  3.874301199857655e-08
epoch  21  loss  3.299542239432185e-08
epoch  22  loss  4.065887431181636e-08
epoch  23  loss  3.9700942799925085e-08
epoch  24  loss  3.5869216574724307e-08
epoch  25  loss  3.2037489461

In [44]:
print('training X:',X_test_sca_tensor.shape)
print('training Y:',y_test_tensor.shape)

print('dataset X:',wine.tensors[0].shape)
print('dataset Y:',wine.tensors[0].shape)

batchX,batchY = next(iter(loader))
print('batch X:',batchX.shape)
print('batch Y:',batchY.shape)

training X: torch.Size([36, 13])
training Y: torch.Size([36])
dataset X: torch.Size([142, 13])
dataset Y: torch.Size([142, 13])
batch X: torch.Size([32, 13])
batch Y: torch.Size([32])


In [54]:
model.eval()

with torch.no_grad():

    output = model(X_test_sca_tensor)

    loss = creterion(output, y_test_tensor)

    predicted_labels = torch.argmax(output, dim=1)

    accuracy = (predicted_labels == y_test_tensor).float().mean()

print(f"Test Loss: {loss.item():.4f}")
print(f"Test Accuracy: {accuracy.item() * 100:.2f}%")

Test Loss: 0.2807
Test Accuracy: 97.22%
